# 03 — 迁移学习：LFP → NCA（冻结与逐级解冻）

## 先把机制说透，再跑代码

**迁移的载体是什么**：就是 `sohnet_lfp.pt` 里的那堆数字——每个卷积核的权重和偏置、每个全连接层的权重矩阵。"把 LFP 的知识搬到 NCA"在代码里只有一行：`model.load_state_dict(torch.load('sohnet_lfp.pt'))`。

**"冻结"冻的是什么**：训练一步 = 前向算预测 → 反向算梯度 → 优化器按梯度更新参数。冻结 = **切断第三步**。被冻的层照常参与前向计算（LFP 学的卷积核照常对 NCA 曲线做卷积），但优化器跳过它，数值一个比特都不变。PyTorch 里两件事共同完成冻结：
1. `param.requires_grad = False` —— 不给它算梯度；
2. 构造优化器时只传入 `requires_grad=True` 的参数。

**BatchNorm 的暗坑（必须处理）**：BN 层有两类量——可学习的 γ/β（受 `requires_grad` 控制）和 running mean/var 统计量（**不受控制**！只要模型处于 `train()` 模式就会被 NCA 数据悄悄改写）。跨域时 BN 统计恰是分布偏移最敏感处。所以冻结骨干时必须**同时** `model.conv.eval()`，把统计量钉在 LFP 值上。下面的训练循环里每个 epoch 都显式做这件事。

**为什么逐级解冻（诚实版论据）**：注意本网络参数 85% 在 FC 头——所以"Stage 1 可训练参数少"这个常见说法在这里**不成立**。真实论据是**风险隔离**：最不该被小样本噪声污染的部分（LFP 学到的通用曲线特征）被保护得最久；最该适应新化学的部分（任务映射）最先放开。头部有 Dropout + weight decay 压制过拟合。

**实验顺序（顺序本身就是论证）**：
1. **Zero-shot**：LFP 模型零微调直接测 NCA —— 量出"特征独自能扛多少"；
2. Stage 1 → 2 → 3 逐级微调 —— 每级的提升量化"迁移学习补了多少"。
这一对对照直接回答教授"特征能否作为 transfer 载体"的问题。

In [ ]:
# ---- Cell 1: 载入行李 + NCA数据 ----
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42); np.random.seed(42)

colmap = json.load(open("colmap.json"))
SEQ_COLS, SCAL_COLS = colmap["SEQ_COLS"], colmap["SCAL_COLS"]
lfp_norm = np.load("norm_lfp.npz")
Y_LO, Y_HI = float(lfp_norm['y_lo']), float(lfp_norm['y_hi'])   # 共享标签空间

nca = pd.read_parquet("nca_features_all.parquet")
cells = sorted(nca['BID'].unique())
print("NCA cells:", cells, "| rows:", len(nca))

# 小样本设定: 少数电池微调, 其余全部盲测(按你的实验设计改)
FT_CELLS   = cells[:2]          # 微调用
TEST_CELLS = cells[2:]          # 盲测
print("fine-tune:", FT_CELLS, "| test:", TEST_CELLS)

## 域独立归一化（Chen 2024 的做法，这里落地）

关键点：**NCA 的 min-max 在 NCA 微调电池上拟合**，不复用 LFP 的 `norm_lfp.npz`。两个域各自映到 [0,1]：LFP 的新鲜电池是 1、NCA 的新鲜电池也是 1——刻度差被消掉，网络看到的是同一种"归一化后的形变语言"。这也意味着 **zero-shot 评估同样用 NCA 自己的归一化**（否则 3 Ah 的 Q 值会远超 LFP 的 [0,1] 范围，网络输入直接爆表——那测的不是特征迁移性，是刻度错配）。

y 不重新拟合：沿用固定的 [50, 105]——两域共享输出坐标系。

In [ ]:
# ---- Cell 2: NCA域内min-max + loaders ----
def part(df, cell_list):
    d = df[df['BID'].isin(cell_list)]
    return (d[SEQ_COLS + SCAL_COLS].to_numpy(np.float32),
            d['SOH'].to_numpy(np.float32), d['BID'].values)

Xft, yft, _ = part(nca, FT_CELLS)
Xte, yte, bid_te = part(nca, TEST_CELLS)

x_min = Xft.min(axis=0); x_max = Xft.max(axis=0)
span = np.where(x_max - x_min < 1e-12, 1.0, x_max - x_min)
norm_x  = lambda X: ((X - x_min) / span).astype(np.float32)
norm_y  = lambda y: ((y - Y_LO) / (Y_HI - Y_LO)).astype(np.float32)
denorm_y = lambda yn: yn * (Y_HI - Y_LO) + Y_LO
np.savez("norm_nca.npz", x_min=x_min, x_max=x_max, y_lo=Y_LO, y_hi=Y_HI)

# 微调集内部再切10%做早停验证
idx = np.random.permutation(len(Xft)); n_va = max(1, len(idx)//10)
va_i, tr_i = idx[:n_va], idx[n_va:]
mk = lambda X, y, sh: DataLoader(TensorDataset(torch.from_numpy(norm_x(X)),
                                               torch.from_numpy(norm_y(y))),
                                 batch_size=128, shuffle=sh)
tr_loader = mk(Xft[tr_i], yft[tr_i], True)
va_loader = mk(Xft[va_i], yft[va_i], False)
te_loader = mk(Xte, yte, False)

In [ ]:
# ---- Cell 3: 网络定义(与NB1完全一致) + 载入LFP权重 ----
class SOHNet(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(3, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(32),
            nn.Conv1d(32, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
            nn.AdaptiveAvgPool1d(4))
        self.head = nn.Sequential(
            nn.Linear(64*4+2, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 32),     nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 1))
    def forward(self, x):
        f = self.conv(x[:, :48].reshape(-1, 3, 16)).flatten(1)
        return self.head(torch.cat([f, x[:, 48:]], 1)).squeeze(-1)

model = SOHNet().to(DEV)
model.load_state_dict(torch.load("sohnet_lfp.pt", map_location=DEV))
print("LFP权重已载入 —— 这一行就是'迁移'本身")

In [ ]:
# ---- Cell 4: 评估函数 + Zero-shot基线(先量特征独自能扛多少) ----
def evaluate(loader):
    model.eval(); ps, ts = [], []
    with torch.no_grad():
        for xb, yb in loader:
            ps.append(denorm_y(model(xb.to(DEV)).cpu().numpy()))
            ts.append(denorm_y(yb.numpy()))
    p, t = np.concatenate(ps), np.concatenate(ts)
    rmse = float(np.sqrt(((p-t)**2).mean()))
    r2 = 1 - ((p-t)**2).sum() / ((t-t.mean())**2).sum()
    return rmse, r2, p, t

results = {}
rmse, r2, _, _ = evaluate(te_loader)
results['zero-shot'] = (rmse, r2)
print(f"ZERO-SHOT (无任何微调): RMSE={rmse:.3f}%  R2={r2:.4f}")
# 预期: 明显差于LFP域内成绩。这个'差'不是失败——它是迁移学习必要性的定量证据,
# 与微调后的恢复量一起构成'特征搭桥+微调补齐'的完整论证。

## 冻结工具箱：三个函数各司其职

- `set_trainable(module, flag)`：批量设置 `requires_grad` —— 控制**梯度是否流向权重**；
- `make_opt(lr)`：只把 `requires_grad=True` 的参数交给 Adam —— 冻结的第二道锁；
- 训练循环里的 `conv.eval()`：当骨干被冻时，把 BN 切到评估模式 —— 钉死 running statistics，堵住第三条"暗改"通道。

三个 Stage 的差异只在"解冻到哪一层 + 学习率多低"。`model.conv` 是 `nn.Sequential`，索引 0–2 是第一 block（Conv+ReLU+BN），3–5 是第二 block，6 是池化——`conv[3:]` 即"最后一个卷积 block"。

In [ ]:
# ---- Cell 5: 冻结工具 + 单阶段训练器 ----
def set_trainable(module, flag):
    for p in module.parameters():
        p.requires_grad = flag

def make_opt(lr, wd=1e-4):
    return torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr, weight_decay=wd)

def run_stage(name, lr, epochs, patience, conv_frozen):
    """conv_frozen=True时: BN统计也一并钉死(conv.eval())"""
    opt = make_opt(lr); lossf = nn.MSELoss()
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n[{name}] lr={lr:g}, 可训练参数 {n_tr}")
    best, bad, best_state = 1e9, 0, None
    for ep in range(1, epochs+1):
        model.train()
        if conv_frozen: model.conv.eval()          # ← BN暗坑的封条
        for xb, yb in tr_loader:
            opt.zero_grad()
            lossf(model(xb.to(DEV)), yb.to(DEV)).backward()
            opt.step()
        v, _, _, _ = evaluate(va_loader)
        if v < best - 1e-4:
            best, bad = v, 0
            best_state = {k: t.clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
        if bad >= patience: break
    model.load_state_dict(best_state)
    rmse, r2, _, _ = evaluate(te_loader)
    results[name] = (rmse, r2)
    print(f"[{name}] TEST RMSE={rmse:.3f}%  R2={r2:.4f}")

In [ ]:
# ---- Cell 6: Stage 1 —— 全骨干冻结, 只训头 ----
# 语义: LFP学的所有滤波器('笔画'与'部件')原封不动;
#       只学'NCA的特征 → NCA的SOH'这一层新映射。
set_trainable(model.conv, False)
set_trainable(model.head, True)
run_stage("stage1_head_only", lr=1e-3, epochs=200, patience=20, conv_frozen=True)

In [ ]:
# ---- Cell 7: Stage 2 —— 解冻最后一个conv block ----
# 语义: 最抽象的'化学签名'层(部件→整字)获准适应NCA的宽峰与斜坡背景;
#       第一block(通用笔画)仍受保护。学习率降一档,避免大步冲毁预训练值。
set_trainable(model.conv[3:], True)      # Conv2+ReLU+BN2
run_stage("stage2_last_block", lr=1e-4, epochs=200, patience=20, conv_frozen=False)

In [ ]:
# ---- Cell 8: Stage 3 —— 全解冻, 极低学习率精修 ----
set_trainable(model, True)
run_stage("stage3_all_lowlr", lr=1e-5, epochs=200, patience=20, conv_frozen=False)
torch.save(model.state_dict(), "sohnet_nca_finetuned.pt")

In [ ]:
# ---- Cell 9: 汇总表 + 逐电池细分 + parity图 ----
print(f"\n{'phase':<20}{'RMSE(%)':>10}{'R2':>10}")
for k, (rm, r2) in results.items():
    print(f"{k:<20}{rm:>10.3f}{r2:>10.4f}")

_, _, p, t = evaluate(te_loader)
print("\n逐电池:")
for b in np.unique(bid_te):
    m = bid_te == b
    rm = float(np.sqrt(((p[m]-t[m])**2).mean()))
    r2b = 1 - ((p[m]-t[m])**2).sum() / ((t[m]-t[m].mean())**2).sum()
    print(f"  cell {b}: RMSE={rm:.3f}%  R2={r2b:.4f}  (n={m.sum()})")

plt.figure(figsize=(5,5))
plt.scatter(t, p, s=5, alpha=0.4)
lo, hi = t.min(), t.max()
plt.plot([lo,hi],[lo,hi],'r--',lw=1)
plt.xlabel('True SOH (%)'); plt.ylabel('Predicted SOH (%)')
plt.title('NCA blind test (after stage 3)')
plt.tight_layout(); plt.savefig('nca_transfer_parity.png', dpi=150); plt.show()

## 结果判读指南

- **zero-shot 差、逐级恢复** → 预期结局：特征搭的桥是通的，微调补齐了域差。汇总表就是答辩的核心证据页。
- **zero-shot 就不差** → 特征跨域强于预期，是加分意外。
- **Stage 3 仍差** → 先查三处：NCA 归一化是否在微调电池上拟合、`HALF_WIN` 是否罩住完整峰形（回 NB2 Cell 7）、h_peak 是否需要改为相对背景定义。框架不动，调提取端超参。

**外部标尺**：Zhao & Wang (ESM 2024) 的 Transformer 路线 LFP→NCA 微调后 RMSE 0.687% / R² 96.8%——数据集不同不可直接比，但"跨化学后的相对衰减幅度"可作参照。